# Do Full-Duplex Speech Models Listen in Text? — v2

**Вопрос пилота.** PersonaPlex (и Moshi) имеет текстовый поток только для *своей* речи. Речь пользователя модель получает исключительно как аудиотокены Mimi. Проверяем: становятся ли слова пользователя декодируемыми как текстовые токены в промежуточных слоях временного трансформера (logit lens через текстовую голову модели)?

**Схема.**
1. Пользовательская речь (LibriSpeech test-clean, склеенная в «реплики» по ~20 с) → токены Mimi → пользовательский поток модели.
2. Свой поток модели: тишина + PAD (модель молча слушает), перед этим — системный промпт, как в PersonaPlex.
3. Один teacher-forced прямой проход, на выходе каждого слоя: `out_norm` → `text_linear` → top-k токенов.
4. Для каждого слова пользователя (тайм-коды из Whisper): попал ли его токен в top-k в окне вокруг слова. Контроль: то же слово в окне из *другой* реплики (уровень случайного совпадения).

**Железо.** Ноутбук сам определяет Kaggle или Colab. Kaggle 2×T4 (модель делится 16+16 слоёв, fp16) или одна GPU ≥ 24 ГБ (L4/A100 в Colab Pro, лучше bf16). Время: ~10–20 мин на загрузку весов, затем секунды на реплику.

**Перед запуском.**
- Для PersonaPlex примите лицензию на https://huggingface.co/nvidia/personaplex-7b-v1 (для Moshi не нужно)
- Kaggle: *Add-ons → Secrets* → `HF_TOKEN`; включите Internet и GPU T4×2.
- Colab: значок ключа слева → секрет `HF_TOKEN` с доступом для ноутбука; *Runtime → Change runtime type* → L4 или A100 (бесплатная T4 на пределе по памяти).

**Что нового в v2.**
- Переключатель модели: `CFG["model"] = "personaplex"` или `"moshi"` (базовый Moshi от Kyutai, лицензию принимать не нужно).
- §12 **Время:** появляется ли слово до своего начала (предсказание), во время или после (узнавание).
- §13 **Смысл или звук:** близки ли ошибочные кандидаты к слову по смыслу (hard ↔ difficult) или по звучанию (know ↔ no).
- §11 Probe с отложенными дикторами и контролем по длительности.
- Сохраняются top-5 по всем слоям (`top5.npz`) — анализы можно пересчитывать без GPU.

**Критерий go/no-go (из v1):** если на каком-то слое hit@5 для реальных окон устойчиво выше контроля (скажем, в 2+ раза при сотнях слов), сигнал есть — идём в полноценный эксперимент.

## 1. Конфигурация

In [ ]:
import os
ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")
BASE = "/kaggle/working" if ON_KAGGLE else ("/content" if ON_COLAB else os.getcwd())
print("platform:", "kaggle" if ON_KAGGLE else "colab" if ON_COLAB else "local", "| base:", BASE)

CFG = dict(
    model        = "personaplex",   # "personaplex" | "moshi"  (run the notebook once per model)
    n_turns      = 40,        # number of ~20 s user turns (40 turns ≈ 2500 words)
    turn_sec     = 20.0,      # target length of one concatenated user turn
    gap_sec      = 0.3,       # silence inserted between LibriSpeech utterances
    silence_frames = 6,       # 0.5 s spacer, as in PersonaPlex (12.5 Hz * 0.5)
    topk         = 50,        # how many lens tokens to keep per layer and frame
    lag_frames   = 4,         # extra frames after word end allowed (4 * 80 ms = 320 ms)
    n_control    = 5,         # random control windows per word
    min_word_chars = 4,       # 'content words' subset: words with >= 4 letters
    save_layers  = [0, 4, 8, 12, 16, 20, 24, 28, 32],  # hidden states kept for probes (0 = embeddings)
    whisper_model = "small.en",
    dtype        = "auto",    # "auto": bf16 if GPU supports it natively (sm80+), else fp16
    seed         = 0,
    save_topk    = 5,          # top-k kept on disk for offline re-analysis
    offsets      = (-8, 20),   # §12: frames relative to word onset/offset (80 ms each)
    sem_topk     = 5,          # §13: candidates per position
    sem_layers   = list(range(0, 33, 2)),
)
PRESETS = {
    # PersonaPlex: system prompt as in its README; user stream carries a sine tone during prompts (as in LMGen)
    "personaplex": dict(model_repo="nvidia/personaplex-7b-v1",
                        text_prompt="You enjoy having a good conversation.", prefix_user="sine"),
    # Moshi was trained without system prompts: short silence on both streams, then the user speaks
    "moshi":       dict(model_repo="kyutai/moshiko-pytorch-bf16", text_prompt="", prefix_user="silence"),
}
CFG.update(PRESETS[CFG["model"]])
CFG["out_dir"] = f"{BASE}/{CFG['model']}_out"
print(CFG["model_repo"], "->", CFG["out_dir"])

## 2. Установка
Не ставим `pip install moshi/.`: пакет PersonaPlex пинит `torch<2.5` и сломает окружение Kaggle. Достаточно добавить исходники в `sys.path`.

In [ ]:
import os, sys, subprocess
os.environ["NO_TORCH_COMPILE"] = "1"   # avoid torch.compile on T4
os.environ["NO_CUDA_GRAPH"] = "1"

if not os.path.exists("personaplex"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/NVIDIA/personaplex.git"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops", "sentencepiece", "sphn", "openai-whisper", "soundfile", "tqdm", "nltk", "sentence-transformers", "scikit-learn"], check=True)
sys.path.insert(0, os.path.abspath("personaplex/moshi"))
os.makedirs(CFG["out_dir"], exist_ok=True)

In [ ]:
import json, math, random, re, tarfile, urllib.request, gc
from pathlib import Path
import numpy as np, pandas as pd, torch
import sentencepiece, soundfile as sf
from scipy.signal import resample_poly
from safetensors import safe_open

from moshi.models import loaders
from moshi.models.lm import LMModel, SILENCE_TOKENS, SINE_TOKENS, _delay_sequence

random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
print(torch.__version__, "GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 3. Загрузка весов
Токен берётся из Kaggle Secrets (или переменной окружения `HF_TOKEN`).

In [ ]:
from huggingface_hub import hf_hub_download
if not os.environ.get("HF_TOKEN"):
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        elif ON_COLAB:
            from google.colab import userdata          # Colab: key icon on the left -> HF_TOKEN, enable notebook access
            os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as e:
        print("could not read HF_TOKEN secret:", e)
assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN (Kaggle Secrets / Colab Secrets / env) after accepting the model license"

repo = CFG["model_repo"]
from huggingface_hub import list_repo_files
files = list_repo_files(repo)
lm_file = loaders.MOSHI_NAME if loaders.MOSHI_NAME in files else next(
    f for f in files if f.endswith(".safetensors") and not f.startswith("tokenizer"))
moshi_path = hf_hub_download(repo, lm_file)
mimi_path  = hf_hub_download(repo, loaders.MIMI_NAME)
tok_path   = hf_hub_download(repo, loaders.TEXT_TOKENIZER_NAME)
sp = sentencepiece.SentencePieceProcessor(tok_path)
print(moshi_path, os.path.getsize(moshi_path) / 1e9, "GB")

## 4. Модель на двух GPU
Грузим веса потензорно (низкое потребление RAM), пропускаем depformer — для logit lens он не нужен. Слои делятся между GPU, проход по слоям делаем вручную.

In [ ]:
SPECIAL_TEXT = [0, 1, 2, 3]   # EPAD, BOS, EOS, PAD
SKIP_PREFIXES = ("depformer", "linears.")

def pick_dtype():
    if CFG["dtype"] != "auto":
        return {"fp16": torch.float16, "bf16": torch.bfloat16}[CFG["dtype"]]
    major = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
    return torch.bfloat16 if major >= 8 else torch.float16

def layer_devices(n_layers):
    n_gpu = torch.cuda.device_count()
    if n_gpu == 0:
        return ["cpu"] * n_layers
    if n_gpu == 1:
        return ["cuda:0"] * n_layers
    half = n_layers // 2
    return ["cuda:0"] * half + ["cuda:1"] * (n_layers - half)

def load_lm_for_lens(path, lm_kwargs, dtype):
    kw = dict(lm_kwargs)
    with torch.device("meta"):
        lm = LMModel(device="meta", dtype=dtype, **kw)
    n_layers = len(lm.transformer.layers)
    devs = layer_devices(n_layers)
    first, last = devs[0], devs[-1]
    expected = set(lm.state_dict().keys())

    def target_device(key):
        m = re.match(r"transformer\.layers\.(\d+)\.", key)
        if m:
            return devs[int(m.group(1))]
        if key.startswith(("emb.", "text_emb.")):
            return first
        return last   # out_norm, text_linear

    sd = {}
    with safe_open(path, framework="pt", device="cpu") as f:
        for key in f.keys():
            if key.startswith(SKIP_PREFIXES) or key not in expected:
                continue
            t = f.get_tensor(key)
            keep_fp32 = key.endswith(".alpha")   # RMSNorm gains are fp32 in Moshi
            sd[key] = t.to(device=target_device(key), dtype=torch.float32 if keep_fp32 else dtype)
    missing = [k for k in expected if k not in sd and not k.startswith(SKIP_PREFIXES)]
    assert not missing, f"missing weights: {missing[:5]}"
    lm.load_state_dict(sd, strict=False, assign=True)
    lm.eval()
    return lm, devs

DTYPE = pick_dtype()
lm_kwargs = dict(loaders._lm_kwargs, dep_q=16)
lm, LAYER_DEVS = load_lm_for_lens(moshi_path, lm_kwargs, DTYPE)
print("dtype:", DTYPE, "| layers:", len(lm.transformer.layers), "| devices:", sorted(set(LAYER_DEVS)))
for i in range(torch.cuda.device_count()):
    print(f"cuda:{i} allocated {torch.cuda.memory_allocated(i)/1e9:.1f} GB")

## 5. Данные: «реплики» пользователя из LibriSpeech test-clean
Склеиваем подряд идущие фразы одной главы до ~20 с. Это чтение, а не разговор — для пилота достаточно; разговорные данные — следующий шаг.

In [ ]:
DATA = Path(f"{BASE}/librispeech")
if not (DATA / "LibriSpeech" / "test-clean").exists():
    DATA.mkdir(parents=True, exist_ok=True)
    tgz = DATA / "test-clean.tar.gz"
    urllib.request.urlretrieve("https://www.openslr.org/resources/12/test-clean.tar.gz", tgz)
    with tarfile.open(tgz) as t:
        t.extractall(DATA)

def librispeech_turns(root, n_turns, turn_sec, gap_sec, sr_out=24000):
    chapters = sorted((root / "LibriSpeech" / "test-clean").glob("*/*"))
    random.shuffle(chapters)
    turns = []
    for ch in chapters:
        trans = {}
        for line in next(ch.glob("*.trans.txt")).read_text().splitlines():
            uid, txt = line.split(" ", 1)
            trans[uid] = txt
        pieces, text, dur = [], [], 0.0
        for uid in sorted(trans):
            wav, sr = sf.read(ch / f"{uid}.flac", dtype="float32")
            wav = resample_poly(wav, sr_out // 1000, sr // 1000).astype(np.float32)  # 16k -> 24k
            pieces += [wav, np.zeros(int(gap_sec * sr_out), np.float32)]
            text.append(trans[uid].lower())
            dur += len(wav) / sr_out + gap_sec
            if dur >= turn_sec:
                break
        turns.append(dict(id=ch.parent.name + "-" + ch.name, wav=np.concatenate(pieces), text=" ".join(text)))
        if len(turns) >= n_turns:
            break
    return turns

turns = librispeech_turns(DATA, CFG["n_turns"], CFG["turn_sec"], CFG["gap_sec"])
print(len(turns), "turns,", sum(len(t["wav"]) for t in turns) / 24000 / 60, "min")

## 6. Пословные тайм-коды (Whisper)
Whisper грузится на GPU, после разметки выгружается.

In [ ]:
import whisper
free_gb = torch.cuda.mem_get_info(0)[0] / 1e9 if torch.cuda.is_available() else 0
wdev = "cuda:0" if free_gb > 3 else "cpu"     # single T4: model leaves too little room, Whisper goes to CPU
print(f"free GPU memory {free_gb:.1f} GB -> whisper on {wdev}")
wmodel = whisper.load_model(CFG["whisper_model"], device=wdev)
for t in turns:
    audio16 = resample_poly(t["wav"], 2, 3).astype(np.float32)   # 24k -> 16k
    res = wmodel.transcribe(audio16, language="en", word_timestamps=True, condition_on_previous_text=False, fp16=(wdev != "cpu"))
    t["words"] = [
        dict(word=re.sub(r"[^\w']", "", w["word"]).lower(), start=float(w["start"]), end=float(w["end"]))
        for seg in res["segments"] for w in seg.get("words", [])
    ]
    t["words"] = [w for w in t["words"] if w["word"]]
del wmodel; gc.collect(); torch.cuda.empty_cache()
print(sum(len(t["words"]) for t in turns), "words; example:", turns[0]["words"][:5])

## 7. Токены Mimi и сборка последовательности
Потоки модели: `0` — свой текст, `1..8` — своё аудио, `9..16` — аудио пользователя. Префикс для PersonaPlex: тишина → системный промпт → тишина (голосовой промпт пропускаем); для Moshi — только тишина. Пока говорит пользователь, свой поток модели — тишина + PAD.

In [ ]:
mimi = loaders.get_mimi(mimi_path, device="cuda:0" if torch.cuda.is_available() else "cpu")
FRAME = int(mimi.sample_rate / mimi.frame_rate)   # 1920 samples = 80 ms
FPS = mimi.frame_rate                              # 12.5

@torch.no_grad()
def mimi_codes(wav):
    pad = (-len(wav)) % FRAME
    x = torch.from_numpy(np.pad(wav, (0, pad))).to(next(mimi.parameters()).device).view(1, 1, -1)
    return mimi.encode(x)[0].cpu()                 # [8, frames]

def wrap_system(text):
    return f"<system> {text.strip()} <system>"

def build_codes(user_codes, prompt_ids, n_sil, prefix_user="sine"):
    # Returns undelayed codes [17, T] and the frame index where the user turn starts.
    pad = 3
    sil = torch.as_tensor(SILENCE_TOKENS).view(8, 1)
    sine = torch.as_tensor(SINE_TOKENS).view(8, 1)
    filler = sine if prefix_user == "sine" else sil
    cols = []
    def frame(text_tok, model_audio, user_audio):
        cols.append(torch.cat([torch.tensor([[text_tok]]), model_audio, user_audio], 0))
    for _ in range(n_sil): frame(pad, sil, filler)
    for tok in prompt_ids:  frame(tok, sil, filler)
    for _ in range(n_sil): frame(pad, sil, filler)
    start = len(cols)
    for i in range(user_codes.shape[1]):
        frame(pad, sil, user_codes[:, i:i + 1])
    return torch.cat(cols, 1).long(), start

prompt_ids = sp.encode(wrap_system(CFG["text_prompt"])) if CFG["text_prompt"] else []
for t in turns:
    t["codes"], t["user_start"] = build_codes(mimi_codes(t["wav"]), prompt_ids, CFG["silence_frames"], CFG["prefix_user"])
print("example sequence:", turns[0]["codes"].shape, "user starts at frame", turns[0]["user_start"])

## 8. Прямой проход с logit lens
Вход строится так же, как в `LMModel.forward_train`: задержки потоков + начальный токен. Позиция `p` видит семантические токены пользователя до кадра `p-1`. На выходе каждого слоя (и после эмбеддингов) применяем `out_norm` + `text_linear`, спецтокены маскируем, сохраняем top-k.

In [ ]:
@torch.no_grad()
def lens_forward(lm, devs, codes, topk, save_layers=()):
    # codes: [17, T] undelayed. Returns topk ids [L+1, T, k] (int32, cpu) and saved hidden states.
    first, last = devs[0], devs[-1]
    codes = codes.unsqueeze(0).to(first)
    initial = lm._get_initial_token().to(first)
    delayed = _delay_sequence(lm.delays, codes, initial)
    seq = torch.cat([initial, delayed], dim=2)[:, :, :-1]
    x = lm.embed_codes(seq)                                   # [1, T, D]
    special = torch.tensor(SPECIAL_TEXT, device=last)

    def lens(h):
        h = h.to(last)
        logits = lm.text_linear(lm.out_norm(h)).float()[0]     # [T, V]
        logits[:, special] = -float("inf")
        return logits.topk(topk, dim=-1).indices.int().cpu()

    tops, saved = [lens(x)], {}
    if 0 in save_layers: saved[0] = x[0].to(torch.float16).cpu()
    for i, layer in enumerate(lm.transformer.layers):
        x = layer(x.to(devs[i]))
        if not torch.isfinite(x).all():
            raise FloatingPointError(f"non-finite activations after layer {i}: try dtype='bf16' on an sm80+ GPU")
        tops.append(lens(x))
        if i + 1 in save_layers: saved[i + 1] = x[0].to(torch.float16).cpu()
    return torch.stack(tops), saved

import time
t0 = time.time()
for k, t in enumerate(turns):
    t["top"], t["hid"] = lens_forward(lm, LAYER_DEVS, t["codes"], CFG["topk"], CFG["save_layers"])
    if k == 0: print(f"first turn: {time.time()-t0:.1f}s, top shape {tuple(t['top'].shape)}")
print(f"done in {time.time()-t0:.0f}s")

## 9. Метрика: слово пользователя в top-k
Для слова в кадрах `[f0, f1)` смотрим позиции `[start+f0+1, start+f1+1+lag]`. Целевые токены — первый SentencePiece-кусок слова в вариантах регистра. Контроль — то же слово в окне той же длины из другой реплики.

In [ ]:
def word_targets(word):
    ids = set()
    for v in {word, word.capitalize(), word.upper()}:
        enc = sp.encode(v)
        if enc:
            ids.add(enc[0])
    return ids - set(SPECIAL_TEXT)

def window(t, w, lag):
    f0 = int(math.floor(w["start"] * FPS)); f1 = int(math.ceil(w["end"] * FPS))
    a = t["user_start"] + f0 + 1
    b = min(t["user_start"] + f1 + 1 + lag, t["top"].shape[1] - 1)
    return a, b

def best_rank(top, a, b, targets):
    # top: [L+1, T, k]. Best (lowest) rank of any target in positions [a, b], per layer; k if absent.
    if b < a or not targets:
        return None
    win = top[:, a:b + 1, :]                                    # [L+1, n, k]
    hit = torch.zeros_like(win, dtype=torch.bool)
    for tid in targets:
        hit |= win == tid
    k = top.shape[2]
    ranks = torch.where(hit, torch.arange(k).view(1, 1, k).expand_as(win), torch.full_like(win, k))
    return ranks.amin(dim=(1, 2)).numpy()                      # [L+1]

def score_all(turns, lag, n_control, seed=0):
    rng = random.Random(seed)
    rows = []
    for ti, t in enumerate(turns):
        others = [u for j, u in enumerate(turns) if j != ti]
        for w in t["words"]:
            targets = word_targets(w["word"])
            a, b = window(t, w, lag)
            r = best_rank(t["top"], a, b, targets)
            if r is None:
                continue
            ctrl = []
            for _ in range(n_control):
                u = rng.choice(others)
                if w["word"] in u["text"].split():
                    continue                                    # control turn must not contain the word
                n = b - a
                lo, hi = u["user_start"] + 1, u["top"].shape[1] - 1 - n
                if hi <= lo:
                    continue
                s = rng.randint(lo, hi)
                ctrl.append(best_rank(u["top"], s, s + n, targets))
            if not ctrl:
                continue
            rows.append(dict(turn=t["id"], word=w["word"], n_chars=len(w["word"]),
                             real=r, ctrl=np.stack(ctrl)))
    return rows

rows = score_all(turns, CFG["lag_frames"], CFG["n_control"], CFG["seed"])
print(len(rows), "scored words")

In [ ]:
def hit_curves(rows, k, min_chars=0):
    sel = [r for r in rows if r["n_chars"] >= min_chars]
    real = np.stack([r["real"] < k for r in sel]).mean(0)
    ctrl = np.stack([(r["ctrl"] < k).mean(0) for r in sel]).mean(0)
    return real, ctrl, len(sel)

curves = []
for k in (1, 5, 20):
    for subset, mc in (("all", 0), ("content", CFG["min_word_chars"])):
        real, ctrl, n = hit_curves(rows, k, mc)
        for layer in range(len(real)):
            curves.append(dict(k=k, subset=subset, layer=layer, real=real[layer], control=ctrl[layer], n_words=n))
curves = pd.DataFrame(curves)
curves.to_csv(f"{CFG['out_dir']}/hit_curves.csv", index=False)

best = curves[(curves.k == 5) & (curves.subset == "content")].assign(ratio=lambda d: d.real / d.control.clip(lower=1e-6))
print(best.sort_values("real", ascending=False).head(8).to_string(index=False))

## 10. Графики

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
for ax, k in zip(axes, (1, 5, 20)):
    for subset, style in (("all", "--"), ("content", "-")):
        d = curves[(curves.k == k) & (curves.subset == subset)]
        ax.plot(d.layer, d.real, style, color="C0", label=f"user word ({subset})")
        ax.plot(d.layer, d.control, style, color="C7", label=f"control ({subset})")
    ax.set_title(f"hit@{k}"); ax.set_xlabel("layer (0 = embeddings)"); ax.grid(alpha=.3)
axes[0].set_ylabel("fraction of words"); axes[0].legend(fontsize=8)
plt.suptitle(f"{CFG['model']}: is the user's word decodable via the text head?")
plt.tight_layout(); plt.savefig(f"{CFG['out_dir']}/hit_curves.png", dpi=150); plt.show()

### Санити-чек: что модель «сказала бы» сама
На последнем слое без маскировки спецтокенов модель, которая слушает, должна почти всегда выдавать PAD. Если нет — вход собран неправильно.

In [ ]:
@torch.no_grad()
def final_argmax(lm, devs, codes):
    first, last = devs[0], devs[-1]
    codes = codes.unsqueeze(0).to(first)
    initial = lm._get_initial_token().to(first)
    seq = torch.cat([initial, _delay_sequence(lm.delays, codes, initial)], 2)[:, :, :-1]
    x = lm.embed_codes(seq)
    for i, layer in enumerate(lm.transformer.layers):
        x = layer(x.to(devs[i]))
    return lm.text_linear(lm.out_norm(x.to(last)))[0].argmax(-1).cpu()

am = final_argmax(lm, LAYER_DEVS, turns[0]["codes"])
user = am[turns[0]["user_start"]:]
print("PAD/EPAD share during user turn:", float(((user == 3) | (user == 0)).float().mean()))
print("non-pad pieces:", [sp.id_to_piece(int(i)) for i in user if int(i) not in SPECIAL_TEXT][:40])

### Примеры: что видно в top-5 на лучшем слое

In [ ]:
L = int(best.sort_values("real", ascending=False).iloc[0].layer)
t = turns[0]
for w in t["words"][:25]:
    a, b = window(t, w, CFG["lag_frames"])
    pieces = []
    for p in range(a, b + 1):
        pieces += [sp.id_to_piece(int(i)) for i in t["top"][L, p, :5]]
    hit = bool(word_targets(w["word"]) & {sp.piece_to_id(x) for x in pieces})
    print(f"{'✓' if hit else ' '} {w['word']:>14s} | {' '.join(dict.fromkeys(pieces))[:110]}")

## 11. Сохранение для probes
Средние скрытые состояния по окну каждого слова на выбранных слоях — для следующего шага (линейные probes на идентичность слова, границы слов, просодию).

In [ ]:
feats = {L: [] for L in CFG["save_layers"]}
meta = []
for t in turns:
    for w in t["words"]:
        a, b = window(t, w, 0)
        if b < a:
            continue
        for L in CFG["save_layers"]:
            feats[L].append(t["hid"][L][a:b + 1].float().mean(0).numpy().astype(np.float16))
        meta.append(dict(turn=t["id"], word=w["word"], start=w["start"], end=w["end"]))
np.savez_compressed(f"{CFG['out_dir']}/word_feats.npz", **{f"layer{L}": np.stack(v) for L, v in feats.items()})
pd.DataFrame(meta).to_csv(f"{CFG['out_dir']}/word_meta.csv", index=False)
json.dump(CFG, open(f"{CFG['out_dir']}/config.json", "w"), indent=2)
print("saved", len(meta), "words x", len(feats), "layers")

### Probe: идентичность слова по слоям
В v1 probe был на 7 словах и ничего не различал. Здесь: все слова с ≥ 5 примерами, PCA до 64, разбиение с **отложенными дикторами** (GroupKFold), контроль — модель только на длительности слова. Ожидание по v1: слово различимо уже на слое 0 (токены Mimi несут фонетику), поэтому probe отвечает на вопрос «есть ли информация», а logit lens — «в каком она виде».

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, GroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline

m = pd.DataFrame(meta)
m["spk"] = m.turn.str.split("-").str[0]
vc = m[m.word.str.len() >= 4].word.value_counts()
classes = vc[vc >= 5].index
mask = m.word.isin(classes).values
y, groups = m.word[mask].values, m.spk[mask].values
print(f"{len(classes)} classes, {mask.sum()} samples, majority {pd.Series(y).value_counts(normalize=True).max():.2f}")
dur = (m.end - m.start)[mask].values[:, None]
base = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)), dur, y,
                       cv=StratifiedKFold(5, shuffle=True, random_state=0)).mean()
print(f"duration-only baseline: {base:.3f}")
probe = []
for L in CFG["save_layers"]:
    X = np.stack(feats[L])[mask].astype(np.float32)
    n_pc = min(64, int(len(y) * 0.8) - 1, X.shape[1])   # PCA must fit inside each training fold
    pipe = make_pipeline(StandardScaler(), PCA(n_pc, random_state=0), LogisticRegression(max_iter=3000, C=0.1))
    acc = cross_val_score(pipe, X, y, cv=GroupKFold(5), groups=groups).mean()
    probe.append(dict(layer=L, acc_speaker_heldout=acc, duration_baseline=base))
    print(f"layer {L:2d}: speaker-held-out acc {acc:.3f}")
pd.DataFrame(probe).to_csv(f"{CFG['out_dir']}/probe_word_identity.csv", index=False)

## 12. Время: предсказание или узнавание?
Привязываем анализ к началу (и отдельно к концу) каждого содержательного слова. В позиции `p = старт_реплики + f0 + 1 + d` модель услышала семантические токены пользователя до кадра `f0 + d` включительно, т. е. примерно `40 + 80·d` мс от начала слова (кадры по 80 мс, точность ±40 мс). Попадание считаем **в одной позиции**, без окна. Контроль — то же слово в случайной позиции другой реплики.

- Попадания при `d < 0` (слово ещё не прозвучало) — **предсказание** следующего слова.
- Рост внутри слова — **узнавание по ходу**; пик после конца — узнавание с задержкой.

In [ ]:
OFFS = np.arange(CFG["offsets"][0], CFG["offsets"][1] + 1)

def hits_at(top, positions, targets, k):
    # top [L+1, T, K]; positions int array; returns [L+1, len(positions)] float with NaN where invalid
    T = top.shape[1]
    out = np.full((top.shape[0], len(positions)), np.nan)
    ok = (positions >= 0) & (positions < T)
    if ok.any():
        win = top[:, positions[ok], :k]
        tt = torch.tensor(sorted(targets), dtype=win.dtype)
        out[:, ok] = torch.isin(win, tt).any(-1).numpy()
    return out

def locked_curves(turns, anchor, k=5, min_chars=4, n_ctrl=3, seed=0):
    rng = random.Random(seed)
    real, ctrl = [], []
    for ti, t in enumerate(turns):
        others = [u for j, u in enumerate(turns) if j != ti]
        for w in t["words"]:
            targets = word_targets(w["word"])
            if len(w["word"]) < min_chars or not targets:
                continue
            f = int(math.floor(w["start"] * FPS)) if anchor == "start" else int(math.ceil(w["end"] * FPS)) - 1
            pos = t["user_start"] + f + 1 + OFFS
            r = hits_at(t["top"], pos, targets, k)
            r[:, pos <= t["user_start"]] = np.nan            # before the user turn started
            cs = []
            for _ in range(n_ctrl):
                u = rng.choice(others)
                if w["word"] in u["text"].split():
                    continue
                lo, hi = u["user_start"] + 1 - OFFS.min(), u["top"].shape[1] - 1 - OFFS.max()
                if hi <= lo:
                    continue
                cs.append(hits_at(u["top"], rng.randint(lo, hi) + OFFS, targets, k))
            if cs:
                real.append(r); ctrl.append(np.nanmean(np.stack(cs), 0))
    return np.stack(real), np.stack(ctrl)

timing = {}
for anchor in ("start", "end"):
    R, C = locked_curves(turns, anchor)
    timing[anchor] = (np.nanmean(R, 0), np.nanmean(C, 0), len(R))
    print(anchor, "-locked:", len(R), "words")
np.savez_compressed(f"{CFG['out_dir']}/timing.npz", offsets=OFFS,
                    **{f"{a}_{s}": v[i] for a, v in timing.items() for i, s in enumerate(("real", "ctrl"))})

In [ ]:
best_layer = int(curves[(curves.k == 5) & (curves.subset == "content")].sort_values("real").iloc[-1].layer)
ms = OFFS * 80 + 40          # audio heard relative to the anchor, frame-quantised (±40 ms)
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for col, anchor in enumerate(("start", "end")):
    real, ctrl, n = timing[anchor]
    diff = real - ctrl
    ax = axes[0, col]
    im = ax.imshow(diff, aspect="auto", origin="lower", cmap="RdBu_r", vmin=-.4, vmax=.4,
                   extent=[ms[0] - 40, ms[-1] + 40, -.5, diff.shape[0] - .5])
    ax.axvline(0, color="k", lw=.8)
    ax.set_title(f"hit@5 (real − control), locked to word {anchor} (n={n})")
    ax.set_xlabel(f"ms of audio heard relative to word {anchor}"); ax.set_ylabel("layer")
    plt.colorbar(im, ax=ax)
    ax = axes[1, col]
    show = dict.fromkeys([best_layer, max(best_layer - 6, 0), real.shape[0] // 2])
    for L, c in zip(show, ("C0", "C1", "C2")):
        ax.plot(ms, real[L], color=c, label=f"layer {L}")
        ax.plot(ms, ctrl[L], color=c, ls=":", alpha=.7)
    ax.axvline(0, color="k", lw=.8); ax.grid(alpha=.3)
    ax.set_xlabel(f"ms relative to word {anchor}"); ax.set_ylabel("hit@5 (dotted = control)"); ax.legend(fontsize=8)
plt.suptitle(f"{CFG['model']}: when does the user's word become readable?")
plt.tight_layout(); plt.savefig(f"{CFG['out_dir']}/timing.png", dpi=150); plt.show()

real, ctrl, _ = timing["start"]
d = real[best_layer] - ctrl[best_layer]
pre = d[ms <= 0]
print(f"layer {best_layer}: excess hit@5 before onset (prediction) mean {np.nanmean(pre):.3f}, max {np.nanmax(pre):.3f}")
half = np.nanmax(d) / 2
print(f"excess reaches half-max at ~{ms[np.argmax(d >= half)]} ms after onset; peak at {ms[np.nanargmax(d)]} ms")

## 13. Смысл или звук?
Для каждого содержательного слова берём кандидатов из top-5 в его окне (только целые слова от 3 букв, исключая само слово и однокоренные с общим префиксом ≥ 4 букв). Для каждого кандидата считаем:
- **смысловую близость** — косинус эмбеддингов `all-MiniLM-L6-v2`;
- **звуковую близость** — 1 − нормированное расстояние Левенштейна между фонемами CMUdict.

Берём максимум по кандидатам и сравниваем с контрольным окном той же длины из другой реплики. Если на поздних слоях растёт смысловой избыток, а звуковой нет, модель переводит речь в значение, а не в написание.

In [ ]:
import nltk
from functools import lru_cache
nltk.download("cmudict", quiet=True)
from nltk.corpus import cmudict
CMU = cmudict.dict()

@lru_cache(maxsize=None)
def phones(w):
    p = CMU.get(w)
    return tuple(re.sub(r"\d", "", x) for x in p[0]) if p else None

def lev(a, b):
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]

@lru_cache(maxsize=None)
def phon_sim(a, b):
    pa, pb = phones(a), phones(b)
    if not pa or not pb:
        return np.nan
    return 1 - lev(pa, pb) / max(len(pa), len(pb))

from sentence_transformers import SentenceTransformer
st_dev = "cuda:0" if torch.cuda.is_available() and torch.cuda.mem_get_info(0)[0] / 1e9 > 1.5 else "cpu"
st_model = SentenceTransformer("all-MiniLM-L6-v2", device=st_dev)
def embed_words(words):
    words = sorted(set(words))
    vecs = st_model.encode(words, batch_size=256, normalize_embeddings=True, show_progress_bar=False)
    return dict(zip(words, vecs))

In [ ]:
CAND = {}                                   # vocab id -> lowercase whole word, for usable pieces only
for i in range(sp.get_piece_size()):
    p = sp.id_to_piece(i)
    if p.startswith("▁") and p[1:].isalpha() and len(p) >= 4 and p[1:].isascii():
        CAND[i] = p[1:].lower()

def related(c, w):
    return c == w or c.startswith(w) or w.startswith(c) or len(os.path.commonprefix([c, w])) >= 4

def window_cands(top, a, b, L, k):
    ids = top[L, a:b + 1, :k].reshape(-1).tolist()
    return {CAND[i] for i in ids if i in CAND}

def sem_phon_rows(turns, layers, k, min_chars=4, seed=0):
    rng = random.Random(seed)
    jobs = []
    for ti, t in enumerate(turns):
        others = [u for j, u in enumerate(turns) if j != ti]
        for w in t["words"]:
            if len(w["word"]) < min_chars or not w["word"].isalpha():
                continue
            a, b = window(t, w, CFG["lag_frames"])
            if b < a:
                continue
            u = rng.choice(others)
            if w["word"] in u["text"].split():
                continue
            lo, hi = u["user_start"] + 1, u["top"].shape[1] - 1 - (b - a)
            if hi <= lo:
                continue
            s = rng.randint(lo, hi)
            jobs.append((w["word"], t, a, b, u, s, s + (b - a)))
    vocab = {j[0] for j in jobs}
    for L in layers:
        for word, t, a, b, u, s, e in jobs:
            vocab |= window_cands(t["top"], a, b, L, k) | window_cands(u["top"], s, e, L, k)
    E = embed_words(vocab)
    rows = []
    for L in layers:
        for word, t, a, b, u, s, e in jobs:
            for cond, cands in (("real", window_cands(t["top"], a, b, L, k)), ("control", window_cands(u["top"], s, e, L, k))):
                cands = sorted(c for c in cands if not related(c, word))
                if not cands:
                    continue
                sims = [float(E[word] @ E[c]) for c in cands]
                phs = [phon_sim(word, c) for c in cands]
                i_sem = int(np.argmax(sims))
                i_ph = None if all(np.isnan(phs)) else int(np.nanargmax(phs))
                rows.append(dict(layer=L, cond=cond, word=word, n_cands=len(cands),
                                 sem=sims[i_sem], sem_cand=cands[i_sem],
                                 phon=np.nan if i_ph is None else phs[i_ph],
                                 phon_cand=None if i_ph is None else cands[i_ph]))
    return pd.DataFrame(rows)

sp_df = sem_phon_rows(turns, CFG["sem_layers"], CFG["sem_topk"])
sp_df.to_csv(f"{CFG['out_dir']}/sem_phon_rows.csv", index=False)
agg = sp_df.groupby(["layer", "cond"])[["sem", "phon", "n_cands"]].mean().unstack("cond")
summary = pd.DataFrame({
    "sem_real": agg[("sem", "real")], "sem_ctrl": agg[("sem", "control")],
    "phon_real": agg[("phon", "real")], "phon_ctrl": agg[("phon", "control")],
    "cands_real": agg[("n_cands", "real")], "cands_ctrl": agg[("n_cands", "control")],
})
summary["sem_excess"] = summary.sem_real - summary.sem_ctrl
summary["phon_excess"] = summary.phon_real - summary.phon_ctrl
summary.to_csv(f"{CFG['out_dir']}/sem_phon_summary.csv")
print(summary.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(summary.index, summary.sem_excess, "-o", label="semantic excess (MiniLM cosine)")
ax.plot(summary.index, summary.phon_excess, "-s", label="phonetic excess (CMUdict)")
ax.axhline(0, color="k", lw=.8); ax.grid(alpha=.3)
ax.set_xlabel("layer"); ax.set_ylabel("max similarity: real − control")
ax.set_title(f"{CFG['model']}: are near-misses similar in meaning or in sound?")
ax.legend(); plt.tight_layout(); plt.savefig(f"{CFG['out_dir']}/sem_phon.png", dpi=150); plt.show()

L = int(summary.sem_excess.idxmax())
ex = sp_df[(sp_df.layer == L) & (sp_df.cond == "real")].sort_values("sem", ascending=False)
print(f"strongest semantic near-misses at layer {L} (word -> candidate):")
for _, r in ex.head(15).iterrows():
    print(f"  {r['word']:>14s} -> {r['sem_cand']:<14s} sem {r['sem']:.2f} | closest in sound: {r['phon_cand']} ({r['phon']:.2f})")

## 14. Сохранение и архив
Топ-5 по всем слоям и разметка реплик — для пересчёта любых анализов без GPU. Всё упаковывается в один zip.

In [ ]:
np.savez_compressed(f"{CFG['out_dir']}/top5.npz",
                    **{t["id"]: t["top"][:, :, :CFG["save_topk"]].numpy().astype(np.int32) for t in turns})
json.dump([dict(id=t["id"], user_start=t["user_start"], text=t["text"], words=t["words"]) for t in turns],
          open(f"{CFG['out_dir']}/turns.json", "w"))
json.dump(CFG, open(f"{CFG['out_dir']}/config.json", "w"), indent=2)

import shutil
from IPython.display import FileLink, display
archive = shutil.make_archive(CFG["out_dir"], "zip", CFG["out_dir"])
print(archive, f"{os.path.getsize(archive) / 1e6:.1f} MB")
for f in sorted(os.listdir(CFG["out_dir"])):
    print(f"  {f:28s} {os.path.getsize(os.path.join(CFG['out_dir'], f)) / 1e6:8.2f} MB")
os.chdir(BASE)
display(FileLink(os.path.basename(archive)))

## Как читать результат и что дальше

- **Сигнал есть**, если на средних/поздних слоях `real` заметно выше `control` (hit@5, content words). Пик по слоям — главный результат пилота: где появляется «неявная транскрипция».
- **Санити-чек** с долей PAD должен быть высоким; иначе — проблема со сборкой входа.
- **Probe** может показать идентичность слова даже там, где logit lens молчит: информация есть, но не в «текстовых координатах».

**Следующие шаги:**
1. Прогон с `CFG["model"] = "moshi"` и сравнение: изменило ли дообучение PersonaPlex то, как модель слушает.
2. Разговорные данные вместо чтения (Fisher/CANDOR или DuplexChat), русский — через FastDialogueSidon.
3. Textless-контроль: маленькая модель без текстового потока + те же probes.
4. Tuned lens вместо logit lens для ранних слоёв.

**Ограничения пилота:** нет голосового промпта; свой поток модели принудительно молчит; LibriSpeech — читаемая речь; тайм-коды Whisper, а не forced alignment.